In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from ipywidgets import interact

# Configurar estilo de gráficos
plt.style.use('seaborn-v0_8-whitegrid')

# Ubicación del archivo de logs
csv_path = "logs/estimation_test.csv"
if not os.path.exists(csv_path):
    csv_path = "robot_ball_log.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Total de registros (ticks): {len(df)}")
else:
    df = None
    print(f"Error: No se encontró el archivo de logs en '{csv_path}'.")
    print("Por favor, ejecuta la simulación en grSim con test_estimation activo antes de correr este notebook.")

Total de registros (ticks): 14595


In [9]:
@interact(
    id=(-1, 3, 1),
    color=(0, 1, 1)
)
def plot_estimation(id=-1, color=0):
    if df is None:
        print("No hay datos para mostrar. Asegúrate de que el archivo de logs exista.")
        return
    
    if id == -1:
        run_data = df[df['robot_id'] == -1]  # Default to robot_id 0 if -1 is selected
    else:
        run_data = df[(df['robot_id'] == id) & (df['color'] == color)]

    # Crear la figura y los ejes
    fig, axs = plt.subplots(3, 1, figsize=(12, 10))

    # Gráfico de posición estimada vs posición real
    axs[0].plot(run_data['tick'], run_data['x'], label='Posición X', color='blue')
    axs[0].set_title(f'Posición Estimada vs Real (X)')
    axs[0].set_xlabel('Tick')
    axs[0].set_ylabel('Posición X')
    axs[0].legend()
    axs[0].grid()

    # Estimate speed using backward difference method
    speed = np.zeros(len(run_data))
    speed[1:] = (run_data['x'].values[1:] - run_data['x'].values[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])

    # Gráfico de velocidad
    axs[1].plot(run_data['tick'], speed, label='Velocidad', color='green')
    axs[1].set_title(f'Velocidad')
    axs[1].set_xlabel('Tick')
    axs[1].set_ylabel('Velocidad')
    axs[1].legend()
    axs[1].grid()

    # Gráfico de aceleración
    acceleration = np.zeros(len(run_data))
    acceleration[1:] = (speed[1:] - speed[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])
    axs[2].plot(run_data['tick'], acceleration, label='Aceleración', color='red')
    axs[2].set_title(f'Aceleración')
    axs[2].set_xlabel('Tick')
    axs[2].set_ylabel('Aceleración')
    axs[2].legend()
    axs[2].grid()

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=-1, description='id', max=3, min=-1), IntSlider(value=0, description='co…

## EKF

In [10]:
from EKF import ExtendedKalmanFilter

# Estado inicial:
# [x, y, sin(theta), cos(theta), vx, vy, omega]
initial_state = np.array([
    0.0,   # x
    0.0,   # y
    0.0,   # sin(theta)
    1.0,   # cos(theta) -> theta = 0
    0.0,   # vx
    0.0,   # vy
    0.0    # omega
])

# Incertidumbre inicial
initial_cov = np.eye(7) * 0.1

# Ruido del modelo
process_noise = np.diag([
    0.01,  # x
    0.01,  # y
    0.001, # sin(theta)
    0.001, # cos(theta)
    0.1,   # vx
    0.1,   # vy
    0.01   # omega
])

# Ruido de las mediciones (x, y, theta)
measurement_noise = np.diag([
    0.05,
    0.05,
    np.deg2rad(5)**2
])

ekf = ExtendedKalmanFilter(
    initial_state,
    initial_cov,
    process_noise,
    measurement_noise
)


In [ ]:

@interact(
    id=(-1, 3, 1),
    color=(0, 1, 1),
    initial_index=(0, 3000, 1),
    window_size=(0, 3000, 1),
    P=(-10, 0, 0.01),
    V=(-10, 0, 0.01),
    Meas=(-10, 0, 0.01)
)
def plot_estimation(id=-1, color=0, initial_index=560, window_size=340, P=-7, V=-4, Meas=-6):
    P_noise    = 10 ** P
    V_noise    = 10 ** V
    Meas_noise = 10 ** Meas
    ekf.set_noise_parameters(P_noise, V_noise, Meas_noise)

    if df is None:
        print("No hay datos para mostrar. Asegúrate de que el archivo de logs exista.")
        return
    
    if id == -1:
        run_data = df[df['robot_id'] == -1]  # Default to robot_id 0 if -1 is selected
    else:
        run_data = df[(df['robot_id'] == id) & (df['team'] == color)]

    x_filtered = run_data['x'].values.copy()

    for i in range(1, len(run_data)):
        dt = run_data['tick'].values[i] - run_data['tick'].values[i - 1]
        dt /= 60
        x_meas = run_data['x'].values[i]
        y_meas = run_data['y'].values[i]
        theta_meas = 0.0
        (
            x,
            *_
        ) = ekf.filter_pose(
            x_meas,
            y_meas,
            theta_meas,
            dt,
        )

        x_filtered[i] = x

    # Crear la figura y los ejes
    fig, axs = plt.subplots(3, 1, figsize=(12, 10))

    cropp = lambda arr, start, window: arr[start:start + window] if window > 0 else arr[start:]

    tick_ax            = cropp(run_data['tick'].values, initial_index, window_size)
    x_raw              = cropp(run_data['x'].values, initial_index, window_size)
    x_filtered_cropped = cropp(x_filtered, initial_index, window_size)

    # Gráfico de posición estimada vs posición real
    axs[0].plot(tick_ax, x_raw, label='Posición X', color='blue')
    axs[0].plot(tick_ax, x_filtered_cropped, label='Posición X Filtrada', color='red', alpha=0.7)
    axs[0].set_title(f'Posición Estimada vs Real (X)')
    axs[0].set_xlabel('Tick')
    axs[0].set_ylabel('Posición X')
    axs[0].legend()
    axs[0].grid()

    # Estimate speed using backward difference method
    speed = np.zeros(len(run_data))
    speed[1:] = (run_data['x'].values[1:] - run_data['x'].values[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])

    speed_filtered = np.zeros(len(run_data))
    speed_filtered[1:] = (x_filtered[1:] - x_filtered[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])

    sp_cropped     = cropp(speed, initial_index, window_size)
    sp_flt_cropped = cropp(speed_filtered, initial_index, window_size)

    # Gráfico de velocidad
    axs[1].plot(tick_ax, sp_cropped, label='Velocidad', color='green')
    axs[1].plot(tick_ax, sp_flt_cropped, label='Velocidad Filtrada', color='red', alpha=0.7)
    axs[1].set_title(f'Velocidad')
    axs[1].set_xlabel('Tick')
    axs[1].set_ylabel('Velocidad')
    axs[1].legend()
    axs[1].grid()

    # Gráfico de aceleración
    acceleration = np.zeros(len(run_data))
    acceleration[1:] = (speed[1:] - speed[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])

    acceleration_filtered = np.zeros(len(run_data))
    acceleration_filtered[1:] = (speed_filtered[1:] - speed_filtered[:-1]) / (run_data['tick'].values[1:] - run_data['tick'].values[:-1])

    ac_cropped     = cropp(acceleration, initial_index, window_size)
    ac_flt_cropped = cropp(acceleration_filtered, initial_index, window_size)

    axs[2].plot(tick_ax, ac_cropped, label='Aceleración', color='red')
    axs[2].plot(tick_ax, ac_flt_cropped, label='Aceleración Filtrada', color='blue', alpha=0.7)
    axs[2].set_title(f'Aceleración')
    axs[2].set_xlabel('Tick')
    axs[2].set_ylabel('Aceleración')
    axs[2].legend()
    axs[2].grid()

    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=-1, description='id', max=3, min=-1), IntSlider(value=0, description='co…